# Country predictions

Load the saved 40-country model and two example images. One example is correct and one is a mistake. Softmax scores are uncalibrated.

In [ ]:
import hashlib
import json

from IPython.display import display
from PIL import Image
import torch
from torchvision import models

from train import REPO_DIR, load_model_checkpoint

torch.set_num_threads(4)
demo_dir = REPO_DIR / "web_demo"
data_dir = demo_dir / "examples"
info = json.loads((demo_dir / "model_info.json").read_text())
checkpoint_path = demo_dir / "model.pt"
with checkpoint_path.open("rb") as stream:
    if hashlib.file_digest(stream, "sha256").hexdigest() != info["model_sha256"]:
        raise ValueError("The checkpoint does not match model_info.json.")
model, checkpoint = load_model_checkpoint(checkpoint_path)
classes = checkpoint["classes"]
if classes != info["classes"]:
    raise ValueError("The checkpoint labels do not match model_info.json.")
preprocess = models.ResNet50_Weights.IMAGENET1K_V1.transforms()
examples = [{"path": item["file"], "country": item["label"]} for item in info["examples"]]

def show_image(index):
    with Image.open(data_dir / examples[index]["path"]) as source:
        image = source.convert("RGB")
    image.thumbnail((1000, 500))
    display(image)

def predict(index):
    row = examples[index]
    with Image.open(data_dir / row["path"]) as source:
        inputs = preprocess(source.convert("RGB")).unsqueeze(0)
    with torch.inference_mode():
        scores = model(inputs).softmax(dim=1)[0]
    values, indices = scores.topk(3)
    predicted_country = classes[int(indices[0])]
    print("Predicted country:", predicted_country)
    print("Dataset label:", row["country"])
    print("Correct:", predicted_country == row["country"])
    print("\nTop three model scores:")
    for value, class_index in zip(values, indices):
        print(f"{classes[int(class_index)]}: {float(value):.1%}")
    return predicted_country

print("Final model loaded on CPU.")
print("Countries:", ", ".join(classes))
print("Two packaged examples are ready.")

## Example image

Set `example_number` to 0 or 1.

In [ ]:
example_number = 0
show_image(example_number)

In [ ]:
predicted_country = predict(example_number)

## Final evaluation

Read the saved holdout metrics and plots.

In [ ]:
results_dir = REPO_DIR / "results/data_growth_20261108"
metrics = json.loads((results_dir / "final_evaluation/metrics.json").read_text())
config = json.loads((results_dir / "final_candidate/config.json").read_text())
print(f"Final holdout accuracy: {metrics['accuracy']:.2%}")
print(f"Final holdout macro F1: {metrics['macro_f1']:.3f}")
print(f"Top-three accuracy: {metrics['top3_accuracy']:.2%}")
print(f"Supported countries: {len(config['classes'])}")
print(f"Final holdout images: {metrics['count']}")
display(Image.open(results_dir / "final_candidate/training_curves.png"))
display(Image.open(results_dir / "final_evaluation/confusion_matrix.png"))